### SageMath version 10.6

In [1]:
os.environ["SAGE_NUM_THREADS"] = '10'
# import logging
# logging.basicConfig()
# logging.getLogger('lefschetz_family.integrator').setLevel(logging.INFO)
# logging.getLogger('lefschetz_family.hypersurface').setLevel(logging.INFO)
# logging.getLogger('lefschetz_family.integrator_simultaneous').setLevel(logging.INFO)

In [2]:
from lefschetz_family import EllipticSurface # lefschetz-family version 0.1.16

# Interface A: integral lattices and periods

This is the basepoint in the $B$-space we use for our computations.

In [3]:
values = [3,6,5,7]

In [4]:
R.<X,Y,Z> = QQ[]
S.<t> = R[]
T.<u> = S[]
B1, B2, B3, B4 = 3+u, 6-u, 5+2*u, 7 +3*u
basepoint = 0
values = [m(basepoint) for m in [B1, B2, B3, B4]]

B12=X
B23=Y
B13 = (B1+B2+B3+B4)*Z - B12 - B23
s2_12 = B1*B2 + B3*B4
s2_23 = B2*B3 + B1*B4
s2_13 = B1*B3 + B2*B4
s3 = B1*B2*B3 + B1*B2*B4 + B1*B3*B4 + B2*B3*B4

P = -81*t*Z^3 + B12*B23*B13 - s2_12*B12*Z^2 - s2_23*B23*Z^2 - s2_13*B13*Z^2 + 2*s3*Z^3

That is the line we choose for monodromy. 

In [5]:
B1, B2, B3, B4

(u + 3, -u + 6, 2*u + 5, 3*u + 7)

And this is the elliptic surface along this line, when $t$ is the elliptic parameter

In [6]:
P

-5*Z^3*u^3 + (-6*X*Z^2 - 2*Y*Z^2 + 25*Z^3)*u^2 + (5*X*Y*Z - 10*X*Z^2 - Y*Z^2 + 27*Z^3)*u - 81*Z^3*t - X^2*Y - X*Y^2 + 21*X*Y*Z + 4*X*Z^2 + 6*Y*Z^2 - 135*Z^3

Below are the forms that we use for our computation of the periods and the action of the order-4 automorphism on the homology lattice. The first four are the holomorphic forms and its derivatives.

In [7]:
forms = []
newforms = [1/P(t=t^4)]
for i in range(3):
    newforms += [newforms[-1].derivative()]
for i in range(4):
    newforms[i] *= P(t=t^4)^(i+1)
    newforms[i] = S(newforms[i](basepoint))
forms += newforms

forms += [t^i for i in range(1,11)]
forms += [t^i*X^3 for i in range(1,11)]

forms = [forms[i] for i in [0,1,2,3,4,5,6,9,10,12,13,14,15,18]]

print(len(forms))
print(forms)

14
[1, -5*X*Y*Z + 10*X*Z^2 + Y*Z^2 - 27*Z^3, (-972*X*Z^5 - 324*Y*Z^5 + 4050*Z^6)*t^4 - 12*X^3*Y*Z^2 + 34*X^2*Y^2*Z^2 - 4*X*Y^3*Z^2 + 102*X^2*Y*Z^3 + 114*X*Y^2*Z^3 + 248*X^2*Z^4 - 382*X*Y*Z^4 + 26*Y^2*Z^4 - 2900*X*Z^5 - 948*Y*Z^5 + 8208*Z^6, 196830*Z^9*t^8 + (34020*X^2*Y*Z^6 + 14580*X*Y^2*Z^6 - 58320*X^2*Z^7 - 248832*X*Y*Z^7 - 1944*Y^2*Z^7 + 381024*X*Z^8 + 47628*Y*Z^8)*t^4 + 390*X^4*Y^2*Z^3 - 210*X^3*Y^3*Z^3 + 150*X^2*Y^4*Z^3 - 720*X^4*Y*Z^4 - 6852*X^3*Y^2*Z^4 - 5166*X^2*Y^3*Z^4 - 24*X*Y^4*Z^4 + 9384*X^3*Y*Z^5 + 39984*X^2*Y^2*Z^5 + 282*X*Y^3*Z^5 + 8880*X^3*Z^6 + 11784*X^2*Y*Z^6 + 19860*X*Y^2*Z^6 + 150*Y^3*Z^6 - 165096*X^2*Z^7 - 351966*X*Y*Z^7 - 8334*Y^2*Z^7 + 798660*X*Z^8 + 141102*Y*Z^8 - 664848*Z^9, t, t^2, t^3, t^6, t^7, t^9, t^10, X^3*t, X^3*t^2, X^3*t^5]


In [8]:
rat = EllipticSurface(P(basepoint), nbits=1000, fibration = [vector(ZZ, [6, 1, -3]), vector(ZZ, [6, 5, 0])])
print("Rational surface")
print("MW:", 10-matrix(rat.trivial_lattice).rank())
print("types:", rat.types)

Rational surface
MW: 2
types: ['I1', 'I1', 'I1', 'I1', 'IV*']


We start by computing the periods and heuristically recovering some invariants of the surface: its Néron-Séveri is $\mathbb Z/6\mathbb Z$, its Picard rank is 14 (and therefore the parabolic homology has rank $22-14+6 =14$), and its discriminant is $2^6$. The Kodaira types of the fibres, the intersection product in some basis, the hyperplane class on this basis and the corresponding numerical approximations of the periods on this basis are certified by the algorithm. The Picard lattice is recovered heuristically from the numerical periods, but we will certify the result in a few cells.

In [9]:
K3 = EllipticSurface(P(basepoint)(t=t^4), nbits=1000, simultaneous_integration=False, fibration=[vector(ZZ, [8, 1, -1]), vector(ZZ, [-4, 9, -2])])
print("K3 surface")
print("types:", K3.types)
# print("MW:", K3.mordell_weil)
# print("Picard rank:", len(K3.neron_severi))
# print("discriminant:", (matrix(K3.neron_severi)*K3.intersection_product*matrix(K3.neron_severi).transpose()).det())

K3 surface
types: ['I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'I1', 'IV*']


In [10]:
print(K3.intersection_product)

[-4 -2 -2 -2 -3 -2 -2 -2 -3 -2 -2 -2 -2 -2 -2 -3 -2 -1 -2 -1| 0  0]
[-2 -2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-3 -1 -1 -1 -4 -2 -2 -2 -3 -2 -2 -2 -2 -2 -2 -3 -2 -1 -2 -1| 0  0]
[-2 -1 -1 -1 -2 -2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -2 -1 -1 -1 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -2 -1 -1 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-3 -1 -1 -1 -3 -1 -1 -1 -4 -2 -2 -2 -2 -2 -2 -3 -2 -1 -2 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -2 -1 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -2 -1 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -2 -1 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -1 -1 -2 -1 -2 -1 -1 -1 -1| 0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -1 -1 -1 -2 -2

In [11]:
print(K3.fibre_class)

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0)


The parabolic lattice is the orthogonal complement of the classes of fibre components, the smooth fibre and the zero section.

In [12]:
parabolic_lattice = (K3.intersection_product * matrix(K3.trivial_lattice).transpose()).left_kernel_matrix()
parabolic_lattice

[ 1  0  0  0  0  0  0  0  0  0  0  0  0  0 -3  1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  1  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  1  0  0  0  0  0  0  0  0  0 -3  1  0  0  0  0  0  0]
[ 0  0  0  0  0  1  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  1  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  1  0  0  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  1  0  0  0  0  0 -3  1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  1  0  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  1  0  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  1  0  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  1  0 -1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  0]

We compute the periods of the chosen forms on the given basis of homology

In [13]:
%time permat = K3.integrate_forms(forms)

CPU times: user 2.56 s, sys: 8.78 s, total: 11.3 s
Wall time: 3min 3s


From the periods, we recover the action of the order 4 

In [14]:
action_on_cohomology = I * diagonal_matrix([1,1,1,1,I,I^2,I^3,I^6,I^7,I^9,I^10,I,I^2,I^5]) 
permat_red = (permat * parabolic_lattice.transpose())
sigma = (permat_red.inverse() * action_on_cohomology * permat_red).change_ring(ZZ)
sigma

[-1  0  0  0 -1  0  0  0 -1  0  0  0  0  0]
[-2 -1 -1 -1 -2 -1 -1 -1 -2 -1 -1 -1 -1 -1]
[ 1  0  0  0  1  0  0  0  1  0  0  0  0  1]
[ 1  0  0  0  1  0  0  0  1  0  0  0  1  0]
[ 1  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 0  0  0  1  0  0  0  0  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0  0  0  0  0  0  0]
[ 0  0  0  0  1  0  0  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  1  0  0  0  0  0  0  0]
[ 0  0  0  0  0  1  0  0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  1  0  0  1  0  0]
[ 0  0  0  0  0  0  0  0  1  0  1  0  0  0]

In [15]:
assert sigma^4==1

$L_-$ is the $-1$-eigenspace of $\sigma^2$. We do a change of basis to make the intersection matrix explicitely $U(2)\oplus U(2)\oplus D_4$.

In [16]:
CBtemp = matrix(ZZ, [[-2, -1, 0, 0, 0, 0, 0, -1], [-1, -1, -1, 0, 0, 0, 0, -1], [2, 2, 0, -1, -1, -1, -1, 0], [0, 1, -1, 0, -1, -1, -1, -1], [1, 1, 0, 0, -1, -1, -1, 0], [-1, -1, 0, 0, 1, 0, 0, 0], [-1, -1, 0, 0, 0, 1, 0, 0], [1, 1, 0, 0, 0, 0, 0, 0]])
assert CBtemp.det() in [-1,1]
Lminus = CBtemp * (sigma^2+1).right_kernel_matrix() * parabolic_lattice
Lminus

[ 1  0  0  0  1 -1 -1 -2 -1  0  0  0  0  0  1  1  0  0  0  0  0  0]
[ 1  1  0  0  1 -1 -1 -1 -1 -1  0  0  0  0  0  1  0  0  0  0  0  0]
[-2  0  1  1  0  1  1  2  2  0 -1 -1 -1 -1 -2  0  0  0  0  0  0  0]
[-1  1  0  1  1  0  0  0  1 -1  0 -1 -1 -1 -1  1  0  0  0  0  0  0]
[-1  0  0  1  0  1  1  1  1  0  0 -1 -1 -1 -1  0  0  0  0  0  0  0]
[ 1  0  0 -1  0  0  0 -1 -1  0  0  1  0  0  1  0  0  0  0  0  0  0]
[ 1  0  0  0  0 -1  0 -1 -1  0  0  0  1  0  1  0  0  0  0  0  0  0]
[-1  0  0  0  0  0  0  1  1  0  0  0  0  0 -1  0  0  0  0  0  0  0]

In [17]:
Lminus*K3.intersection_product*Lminus.transpose()

[ 0  2  0  0  0  0  0  0]
[ 2  0  0  0  0  0  0  0]
[ 0  0  0  2  0  0  0  0]
[ 0  0  2  0  0  0  0  0]
[ 0  0  0  0 -2  0  0  1]
[ 0  0  0  0  0 -2  0  1]
[ 0  0  0  0  0  0 -2  1]
[ 0  0  0  0  1  1  1 -2]

The Picard lattice is $L_+$, the orthogonal complement of $L_-$. We check that this coincides with the heuristic computation done by `lefschetz-family`

In [18]:
assert Lminus.rank() +  len(K3.neron_severi) == 22
Lminus*K3.intersection_product * matrix(K3.neron_severi).transpose()

[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0]

We identify the Picard lattice as the only 2-elementary lattice with invariants $r=14=1+13$, $l=6$, and $\delta=0$, that is $U\oplus D_4\oplus D_4\oplus D_4$.

In [19]:
IntegralLattice(matrix(K3.neron_severi) * K3.intersection_product * matrix(K3.neron_severi).transpose()).discriminant_group()

Finite quadratic module over Integer Ring with invariants (2, 2, 2, 2, 2, 2)
Gram matrix of the quadratic form with values in Q/2Z:
[  1   0   0 1/2   0   0]
[  0   1 1/2 1/2   0 1/2]
[  0 1/2   1   0 1/2 1/2]
[1/2 1/2   0   1 1/2 1/2]
[  0   0 1/2 1/2   1   0]
[  0 1/2 1/2 1/2   0   1]

In [20]:
IntegralLattice("U").direct_sum(IntegralLattice("U")).twist(2).direct_sum(IntegralLattice("D4")).discriminant_group()

Finite quadratic module over Integer Ring with invariants (2, 2, 2, 2, 2, 2)
Gram matrix of the quadratic form with values in Q/2Z:
[  0 1/2   0   0   0   0]
[1/2   0   0   0   0   0]
[  0   0   1 1/2 1/2   0]
[  0   0 1/2   0   0   0]
[  0   0 1/2   0   1 1/2]
[  0   0   0   0 1/2   1]

We now find the $i$-eigenspace of $\sigma$. For this we do a further change of basis to make $\sigma$ in a block-triangular shape

In [21]:
CBtemp = matrix(ZZ, [[0, 1, 0, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, -1, 0, 0, 0, 0], [1, 1, 0, -1, 2, 1, 1, 2], [0, 1, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, -1, 0], [0, -1, 0, 0, -1, -1, 0, -1], [0, 0, 0, 0, 0, 0, 0, -1]])
assert CBtemp.det() in [-1,1]
Lminus = CBtemp * Lminus
Lminus

[ 1  1  0  0  1 -1 -1 -1 -1 -1  0  0  0  0  0  1  0  0  0  0  0  0]
[-1  1  1  1  1  0  0  1  1 -1 -1 -1 -1 -1 -2  1  0  0  0  0  0  0]
[ 1 -1  0 -1 -1  0  0  0 -1  1  0  1  1  1  1 -1  0  0  0  0  0  0]
[ 1  0  0  0  1 -1  0 -1 -1  0  0  0  0 -1  0  1  0  0  0  0  0  0]
[ 0  1  0  1  1  0  0  0  0 -1  0 -1 -1 -1 -1  1  0  0  0  0  0  0]
[-1  0  0  0  0  1  0  1  1  0  0  0 -1  0 -1  0  0  0  0  0  0  0]
[ 0 -1  0  0 -1  0  0  0  0  1  0  0  1  1  1 -1  0  0  0  0  0  0]
[ 1  0  0  0  0  0  0 -1 -1  0  0  0  0  0  1  0  0  0  0  0  0  0]

In [22]:
QT = Lminus * K3.intersection_product * Lminus.transpose()
QT

[ 0  0  0  2  0  0  0  0]
[ 0  0 -2  0  0  0  0  0]
[ 0 -2  0  0  0  0  0  0]
[ 2  0  0  0  0  0  0  0]
[ 0  0  0  0 -2  0  1 -1]
[ 0  0  0  0  0 -2  1  1]
[ 0  0  0  0  1  1 -2  0]
[ 0  0  0  0 -1  1  0 -2]

In [23]:
action_on_Lminus = parabolic_lattice.solve_left(Lminus).transpose().solve_right(sigma * parabolic_lattice.solve_left(Lminus).transpose())
action_on_Lminus

[ 0 -1  0  0  0  0  0  0]
[ 1  0  0  0  0  0  0  0]
[ 0  0  0 -1  0  0  0  0]
[ 0  0  1  0  0  0  0  0]
[ 0  0  0  0  0 -1  0  0]
[ 0  0  0  0  1  0  0  0]
[ 0  0  0  0  0  0  0 -1]
[ 0  0  0  0  0  0  1  0]

The $i$-eigenspace and its hermitian form is then easily obtainable

In [24]:
IEigenspace = matrix([Lminus[i] - I*Lminus[i+1] for i in [0,2,4,6]])
IEigenspace[0] *=-I
Hermitian_form = IEigenspace.conjugate() * K3.intersection_product * IEigenspace.transpose()
Hermitian_form

[       0        4        0        0]
[       4        0        0        0]
[       0        0       -4  2*I + 2]
[       0        0 -2*I + 2       -4]

We end this interface with a check that $L_+$ is the full generic Picard lattice. We do this by certifying that no elements of the $i$-eigenspace lies generically in the Picard lattice, or equivalently, that the holomorphic form does not identically vanish on any cycle, which amounts to checking that the determinant is nonzero. Note that the first four elements of `forms` are the first four derivatives of the holomorphic form.

In [25]:
frobenius_period_on_IEigenspace = diagonal_matrix([1/factorial(k) for k in range(4)]) * permat.submatrix(0,0,4) * IEigenspace.transpose()
frobenius_period_on_IEigenspace.det()

[2.4840369352531565149954317346639247298535128519645008708431918816072412038682662264968290709189540870476164140612471040603634837601613578679286449821894977544448843411070966619988277304221429765292938963835478940066242212197410272224708311981571225647606866851587609057 +/- 4.22e-269] + [2.4840369352531565149954317346639247298535128519645008708431918816072412038682662264968290709189540870476164140612471040603634837601613578679286449821894977544448843411070966619988277304221429765292938963835478940066242212197410272224708311981571225647606866851587609057 +/- 4.22e-269]*I

# Interface B: Monodromy computation

We start by computing the Picard-Fuchs equation that we will use to compute monodromy

In [26]:
from ore_algebra import OreAlgebra # ore_algebra version 0.5
from lefschetz_family.voronoi import FundamentalGroupVoronoi
from lefschetz_family.integrator import Integrator
from lefschetz_family.numperiods.integerRelations import IntegerRelations

In [27]:
try:
    mathematica('Needs["RISC`HolonomicFunctions`"]')
except:
    pass
mathematica('Needs["RISC`HolonomicFunctions`"]')

Null

In [28]:
%%time
Qu.<u> = QQ[]
Su.<Du> = OreAlgebra(Qu)
mathematica("f="+str(P(t=t^4)))
mathematica("R = 1/f")
mathematica("ann = Annihilator[R, {Der[X], Der[Y], Der[Z], Der[t], Der[u]}]")
mathematica("ann1 = CreativeTelescoping[ann , Der[X], { Der[Y], Der[Z], Der[t], Der[u] }][[1]]")
mathematica("ann2 = CreativeTelescoping[ann1, Der[Y], { Der[Z], Der[t], Der[u] }][[1]]")
mathematica("ann3 = CreativeTelescoping[ann2, Der[Z], { Der[t], Der[u] }][[1]]")
mathematica("annfin = CreativeTelescoping[ann3, Der[t], { Der[u] }][[1]]")
Llist = mathematica("annfin[[1]][[1]]").sage()
PicardFuchsEquation = sum([Qu(str(c)) * Su.gen()**ZZ(p[0]) for c, p in Llist])
PicardFuchsEquation

CPU times: user 8.3 ms, sys: 9.86 ms, total: 18.2 ms
Wall time: 12.5 s


(11695962877185792*u^16 - 31776552306493568*u^15 - 6363087420043668480*u^14 - 67203875932241772672*u^13 - 147637037607940140544*u^12 + 1589864711310438427008*u^11 + 13003354132080448826368*u^10 + 39118681488174725906816*u^9 + 22374005366024884346880*u^8 - 222710154990270978797952*u^7 - 839206231334150205170688*u^6 - 1497639830693035485813120*u^5 - 1471598194631883276427776*u^4 - 633570169020372245467008*u^3 + 154511939791691662795776*u^2 + 272266892971306763499648*u + 79579682645688807563520)*Du^4 + (152047517403415296*u^15 - 497657271351852064*u^14 - 92263979873279508864*u^13 - 1047099003370104817120*u^12 - 3813634402103151706112*u^11 + 7780291969035760541408*u^10 + 121986685125872703181696*u^9 + 480670236010102569843744*u^8 + 859394147042315387815680*u^7 + 73090966026788532660384*u^6 - 3123080043946412655620736*u^5 - 7293904956366880317427872*u^4 - 8418986637523918294878720*u^3 - 5258092230052164356807520*u^2 - 1512631839192514626805632*u - 81788133455339304113568)*Du^3 + (5131603712

We compute a basis of the fundamental group and the action of monodromy on the Frobenius basis at the base point.

In [29]:
roots = PicardFuchsEquation.leading_coefficient().roots(QQbar, multiplicities=False)
fg = FundamentalGroupVoronoi(roots, basepoint)
_ = fg.sort_loops()
integrator = Integrator(fg, PicardFuchsEquation, 1200)
monodromy_on_frobenius = integrator.transition_matrices

The Frobenius period matrix serves as base change to recover the action on the $i$-eigenspace.

In [30]:
integration_correction = diagonal_matrix([1/factorial(i) for i in range(4)])
homology_to_frobenius = integration_correction * frobenius_period_on_IEigenspace

We recover the Gaussian integer matrices. The bounds below (of more than 250 decimal digits) show that the recovered integers are certified.

In [31]:
Ms = []
for tM in monodromy_on_frobenius:
    M = (homology_to_frobenius.inverse() * tM * homology_to_frobenius)
    Ms += [M.apply_map(lambda c: c.real()).change_ring(ZZ) + I*M.apply_map(lambda c: c.imag()).change_ring(ZZ)]
    print(M - Ms[-1])

[[+/- 3.64e-267] + [+/- 3.64e-267]*I [+/- 8.27e-267] + [+/- 8.27e-267]*I [+/- 7.13e-267] + [+/- 7.13e-267]*I [+/- 3.79e-267] + [+/- 3.79e-267]*I]
[[+/- 1.45e-267] + [+/- 1.45e-267]*I [+/- 3.28e-267] + [+/- 3.28e-267]*I [+/- 2.83e-267] + [+/- 2.83e-267]*I [+/- 1.50e-267] + [+/- 1.50e-267]*I]
[[+/- 6.26e-268] + [+/- 6.26e-268]*I [+/- 1.43e-267] + [+/- 1.43e-267]*I [+/- 1.23e-267] + [+/- 1.23e-267]*I [+/- 6.51e-268] + [+/- 6.51e-268]*I]
[[+/- 5.43e-268] + [+/- 5.43e-268]*I [+/- 1.23e-267] + [+/- 1.23e-267]*I [+/- 1.07e-267] + [+/- 1.07e-267]*I [+/- 5.60e-268] + [+/- 5.60e-268]*I]
[[+/- 1.74e-267] + [+/- 1.74e-267]*I [+/- 3.30e-267] + [+/- 3.30e-267]*I [+/- 3.00e-267] + [+/- 3.00e-267]*I [+/- 1.23e-267] + [+/- 1.23e-267]*I]
[[+/- 6.85e-268] + [+/- 6.85e-268]*I [+/- 1.31e-267] + [+/- 1.31e-267]*I [+/- 1.19e-267] + [+/- 1.19e-267]*I [+/- 4.83e-268] + [+/- 4.83e-268]*I]
[[+/- 2.98e-268] + [+/- 2.98e-268]*I [+/- 5.66e-268] + [+/- 5.66e-268]*I [+/- 5.16e-268] + [+/- 5.16e-268]*I [+/- 2.11e-268]

As a sanity check, we can verify that they respect the Hermitian form

In [32]:
all([M.transpose().conjugate() * Hermitian_form* M == Hermitian_form for M in Ms])

True

We list the non-trivial monodromy matrices, along with the corresponding loop in $u$-space, and the values of the $B$s at the corresponding critical value.

In [33]:
for M, critical_value, loop in zip(Ms, fg.points[1:], fg.pointed_loops_vertices):
    if M==1:
        continue
    print("critical value:",critical_value)
    print("minimal polynomial of critical value:",critical_value.minpoly())
    print("value of Bs at critical value:", [m(critical_value) for m in [B1, B2, B3, B4]])
    print("loop:", loop.path)
    print(M)
    # print("-"*60)
    print("\n"*2)

critical value: -0.7055665131205157?
minimal polynomial of critical value: x^4 - 276/409*x^3 - 2666/409*x^2 - 340/409*x + 889/409
value of Bs at critical value: [2.294433486879485?, 6.705566513120516?, 3.588866973758969?, 4.883300460638453?]
loop: [0, -33/19*I - 6/17, 33/19*I - 6/17, 31/21*I - 32/53, -27/22, -31/21*I - 32/53, -33/19*I - 6/17, 0]
[     1      2  I - 1 -I + 1]
[     0      1      0      0]
[     0      2      I -I + 1]
[     0   -2*I  I + 1     -I]



critical value: -2.052389311019607?
minimal polynomial of critical value: x^4 - 276/409*x^3 - 2666/409*x^2 - 340/409*x + 889/409
value of Bs at critical value: [0.9476106889803931?, 8.05238931101961?, 0.8952213779607861?, 0.842832066941180?]
loop: [0, -33/19*I - 6/17, -31/21*I - 32/53, -13/15*I - 9/5, -4/15*I - 17/9, -177/106, 4/15*I - 17/9, 5/24*I - 57/26, -5/24*I - 57/26, -4/15*I - 17/9, -13/15*I - 9/5, -31/21*I - 32/53, -33/19*I - 6/17, 0]
[       3        2    I - 3        2]
[       2        3    I - 3        2]
[     

# Interface C: Discriminant computations and 

In [34]:
discriminant_group = IntegralLattice(Lminus * K3.intersection_product * Lminus.transpose()).discriminant_group()
CBdiscriminant = matrix(ZZ, [[0, 0, 0, 0, 1, -1], [0, 0, 1, -1, 0, 0], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, 0, -1, 0, 0], [0, 0, 0, 0, 0, -1]])
CBdiscriminant * matrix([g.lift() for g in discriminant_group.gens()])

[ 1/2    0    0    0    0    0    0    0]
[   0  1/2    0    0    0    0    0    0]
[   0    0  1/2    0    0    0    0    0]
[   0    0    0  1/2    0    0    0    0]
[   0    0    0    0  1/2 -1/2    0    0]
[   0    0    0    0    0    0  1/2 -1/2]

In [35]:
CBdiscriminant * discriminant_group.gram_matrix_quadratic() * CBdiscriminant.transpose()

[  2   0   0 1/2   0   1]
[  0   2 1/2   0   1   0]
[  0 1/2   0   0   0   0]
[1/2   0   0   0   0   0]
[  0   1   0   0   1 1/2]
[  1   0   0   0 1/2   1]

In [36]:
Lminus_to_eigenspaces = Lminus.solve_left(IEigenspace).stack(Lminus.solve_left(IEigenspace).conjugate())

Checking that monodromy preserves the intersection product on the transcendental lattice

In [37]:
monodromy_on_Lminus = [Lminus_to_eigenspaces.transpose() * block_diagonal_matrix([M, M.conjugate()]) * Lminus_to_eigenspaces.transpose().inverse() for M in Ms]
# print([M.transpose() * TrXlat.gram_matrix() * M == TrXlat.gram_matrix() for M in monodromy_on_TrX])

In [38]:
[M for M in monodromy_on_Lminus if M!=1]

[
[ 1  0  0  2  1 -1 -1  1]  [ 3  0  0  2  1 -3  0  2]
[ 0  1 -2  0  1  1 -1 -1]  [ 0  3 -2  0  3  1 -2  0]
[ 0  0  1  0  0  0  0  0]  [ 0 -2  3  0 -3 -1  2  0]
[ 0  0  0  1  0  0  0  0]  [ 2  0  0  3  1 -3  0  2]
[ 0  0  2  0  0 -1  1  1]  [ 0 -4  4  0 -5 -2  4  0]
[ 0  0  0  2  1  0 -1  1]  [ 4  0  0  4  2 -5  0  4]
[ 0  0  0  2  1 -1  0  1]  [ 2  0  0  2  1 -3  1  2]
[ 0  0 -2  0  1  1 -1  0], [ 0  2 -2  0  3  1 -2  1],

[ 1  0 -1  1  1 -1  0  1]  [ 2  1 -1  1  1 -1 -1  0]
[ 0  1 -1 -1  1  1 -1  0]  [-1  2 -1 -1  1  1  0 -1]
[ 0  0  1  0  0  0  0  0]  [ 1 -1  2  1 -1 -1  0  1]
[ 0  0  0  1  0  0  0  0]  [ 1  1 -1  2  1 -1 -1  0]
[ 0  0  1  1  0 -1  1  0]  [ 3 -1  1  3  0 -3 -1  2]
[ 0  0 -1  1  1  0  0  1]  [ 1  3 -3  1  3  0 -2 -1]
[ 0  0  0  0  0  0  1  0]  [ 2  2 -2  2  2 -2 -1  0]
[ 0  0  0  0  0  0  0  1], [-2  2 -2 -2  2  2  0 -1],

[ 1  0  0  0  0  0  0  0]  [ 1  0  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0]  [ 0  1  0  0  0  0  0  0]
[ 1 -1  1  0 -1 -1  1  0]  [ 0  0  1  0 

In [39]:
all([M.transpose()*QT*M==QT for M in monodromy_on_Lminus])

True

The monodromy action on the discriminant group is $(\mathbb Z/2\mathbb Z)^4$, generated by the monodromy around the $B_i=0$ divisors

In [40]:
monodromy_on_discriminant = []
for M in monodromy_on_Lminus:
    if M==1:
        continue
    monodromy_on_discriminant += [matrix(IntegerModRing(2), [discriminant_group(M*v.lift()) for v in discriminant_group.gens()])]
monodromy_on_discriminant = [CBdiscriminant*M*CBdiscriminant.change_ring(IntegerModRing(2)).inverse() for M in monodromy_on_discriminant]
monodromy_on_discriminant

[
[1 0 0 0 0 0]  [1 0 0 0 0 0]  [1 0 0 0 0 0]  [0 1 1 1 1 0]
[0 1 0 0 0 0]  [0 1 0 0 0 0]  [0 1 0 0 0 0]  [1 0 1 1 1 0]
[0 0 1 0 0 0]  [0 0 1 0 0 0]  [1 1 1 0 1 0]  [1 1 0 1 1 0]
[0 0 0 1 0 0]  [0 0 0 1 0 0]  [1 1 0 1 1 0]  [1 1 1 0 1 0]
[0 0 0 0 1 0]  [0 0 0 0 1 0]  [0 0 0 0 1 0]  [0 0 0 0 1 0]
[0 0 0 0 0 1], [0 0 0 0 0 1], [1 1 0 0 1 1], [1 1 1 1 1 1],

[1 0 1 1 1 0]  [1 0 0 0 0 0]  [1 0 0 0 0 0]  [1 0 0 0 0 0]
[0 1 1 1 1 0]  [0 1 0 0 0 0]  [0 1 0 0 0 0]  [0 1 0 0 0 0]
[0 0 1 0 0 0]  [0 0 1 0 0 0]  [0 0 1 0 0 0]  [0 0 1 0 0 0]
[0 0 0 1 0 0]  [0 0 0 1 0 0]  [0 0 0 1 0 0]  [0 0 0 1 0 0]
[0 0 0 0 1 0]  [0 0 0 0 1 0]  [0 0 0 0 1 0]  [0 0 0 0 1 0]
[0 0 1 1 1 1], [0 0 0 0 1 1], [0 0 0 0 0 1], [0 0 0 0 0 1]
]

In [41]:
MatrixGroup(monodromy_on_discriminant).group_id()

[16, 14]

### Computing the action of area permutations on the transcendental lattice

In [42]:
R.<X,Y,Z> = QQ[]
S.<t> = R[]
T.<u> = S[]

In [43]:
def derivative(P, A, v):
    k = ZZ((A(1)(1).degree()+3)/3)
    newnum = 1/k*P*A.derivative(v) - A*P.derivative(v)
    return newnum
    
def swap_action(B1, B2, B3, B4):
    basepoint = 0
    values = [m(basepoint) for m in [B1, B2, B3, B4]]
    print("Bs:", B1,",", B2,",", B3,",", B4)
    
    B12, B23 = X, Y
    B13 = (B1+B2+B3+B4)*Z - B12 - B23
    s2_12 = B1*B2 + B3*B4
    s2_23 = B2*B3 + B1*B4
    s2_13 = B1*B3 + B2*B4
    s3 = B1*B2*B3 + B1*B2*B4 + B1*B3*B4 + B2*B3*B4
    P = -81*t*Z^3 + B12*B23*B13 - s2_12*B12*Z^2 - s2_23*B23*Z^2 - s2_13*B13*Z^2 + 2*s3*Z^3
    P
    
    Qu.<u> = QQ[]
    Su.<Du> = OreAlgebra(Qu)
    mathematica("f="+str(P(t=t^4)))
    mathematica("R = 1/f")
    mathematica("ann = Annihilator[R, {Der[X], Der[Y], Der[Z], Der[t], Der[u]}]")
    mathematica("ann1 = CreativeTelescoping[ann , Der[X], { Der[Y], Der[Z], Der[t], Der[u] }][[1]]")
    mathematica("ann2 = CreativeTelescoping[ann1, Der[Y], { Der[Z], Der[t], Der[u] }][[1]]")
    mathematica("ann3 = CreativeTelescoping[ann2, Der[Z], { Der[t], Der[u] }][[1]]")
    mathematica("annfin = CreativeTelescoping[ann3, Der[t], { Der[u] }][[1]]")
    Llist = mathematica("annfin[[1]][[1]]").sage()
    L = sum([Qu(str(c)) * Su.gen()**ZZ(p[0]) for c, p in Llist])
    
    ders = [P.parent()(1)]
    for i in range(3):
        ders += [derivative(P(t=t^4), ders[-1], P.parent().gen())]
    ders = [v(basepoint) for v in ders]
    roots = L.leading_coefficient().roots(QQbar, multiplicities=False) + [1]
    fg = FundamentalGroupVoronoi(roots, basepoint)
    _ = fg.sort_loops()
    path = [fg.vertices[v] for v in fg.paths[fg.points.index(1)-1]] + [1]
    print("Picard-Fuchs equation:", L)
    print("chosen u-path:", path)
    swap = L.numerical_transition_matrix(path, eps=2^-1200)
    pertemp = K3.integrate_forms(ders)
    pertemp2 = diagonal_matrix([1,-1,1,-1]) * pertemp
    period_on_IEigenspace = pertemp * IEigenspace.transpose()
    period_on_IEigenspace2 = pertemp2 * IEigenspace.transpose()
    
    integration_correction = diagonal_matrix([1/factorial(i) for i in range(4)])
    homology_to_frobenius = integration_correction * period_on_IEigenspace
    homology_to_frobenius2 = integration_correction * period_on_IEigenspace2
    swap = homology_to_frobenius2.inverse() * swap * homology_to_frobenius
    swap = swap.apply_map(lambda c: c.real()).change_ring(ZZ) + I * swap.apply_map(lambda c: c.imag()).change_ring(ZZ)
    return swap

In [44]:
%%time
B1, B2, B3, B4 = vector(T, [3+3*u, 6-3*u, 5, 7])
swapB1B2 = swap_action(B1,B2,B3,B4)

Bs: 3*u + 3 , -3*u + 6 , 5 , 7
Picard-Fuchs equation: (53747712*u^11 - 295612416*u^10 + 1545744384*u^9 - 4738756608*u^8 + 38365387776*u^7 - 114233946624*u^6 + 202730118144*u^5 - 230819465472*u^4 - 95511031520*u^3 + 319684791888*u^2 - 157318296240*u + 20268659488)*Du^4 + (564350976*u^10 - 2821754880*u^9 + 15918529536*u^8 - 46743588864*u^7 + 509274210816*u^6 - 1376071441920*u^5 + 2120637657984*u^4 - 1996713590016*u^3 + 38821149072*u^2 + 737134477296*u - 247077369588)*Du^3 + (1427673600*u^9 - 6424531200*u^8 + 36779111424*u^7 - 98745744384*u^6 + 1701930346560*u^5 - 4022952078240*u^4 + 5155683722496*u^3 - 3755663356896*u^2 + 1180133318874*u - 96084231117)*Du^2 + (772623360*u^8 - 3090493440*u^7 + 16248231936*u^6 - 37927968768*u^5 + 1258997510016*u^4 - 2458387314432*u^3 + 2450357192448*u^2 - 1226969781120*u + 433692664476)*Du + 20995200*u^7 - 73483200*u^6 + 231530400*u^5 - 395118000*u^4 + 52793645400*u^3 - 78832091700*u^2 + 39373545150*u - 6559511625
chosen u-path: [0, -53/52*I + 2/19, -54/55

In [45]:
%%time
B1, B2, B3, B4 = vector(T, [3+2*u, 6, 5-2*u, 7])
swapB1B3 = swap_action(B1,B2,B3,B4)

Bs: 2*u + 3 , 6 , -2*u + 5 , 7
Picard-Fuchs equation: (131072*u^11 - 720896*u^10 - 2686976*u^9 + 17498112*u^8 + 1777195520*u^7 - 6306888448*u^6 + 32892680640*u^5 - 66420047072*u^4 - 61096919312*u^3 + 154899235064*u^2 - 80445123024*u + 12342822660)*Du^4 + (1376256*u^10 - 6881280*u^9 - 51494912*u^8 + 247267328*u^7 + 26492675328*u^6 - 80372363008*u^5 + 323861541856*u^4 - 513466904256*u^3 - 134818529000*u^2 + 378113311688*u - 114027224556)*Du^3 + (3481600*u^9 - 15667200*u^8 - 140138496*u^7 + 563598336*u^6 + 99645992736*u^5 - 250560534480*u^4 + 732914502228*u^3 - 848518974894*u^2 + 299430833736*u - 16661546783)*Du^2 + (1884160*u^8 - 7536640*u^7 - 49739264*u^6 + 175596032*u^5 + 80369260000*u^4 - 161039972800*u^3 + 314391369772*u^2 - 233840861260*u + 146878516114)*Du + 51200*u^7 - 179200*u^6 + 1286400*u^5 - 2768000*u^4 + 3519227200*u^3 - 5276162400*u^2 + 2637752000*u - 439603600
chosen u-path: [0, -89/44*I + 2/13, -195/97*I + 37/92, -195/97*I + 52/87, -89/44*I + 11/13, 1]
CPU times: user 1.73

In [46]:
%%time
B1, B2, B3, B4 = vector(T, [3+4*u, 6, 5, 7-4*u])
swapB1B4 = swap_action(B1,B2,B3,B4)

Bs: 4*u + 3 , 6 , 5 , -4*u + 7
Picard-Fuchs equation: (8589934592*u^11 - 47244640256*u^10 + 122540785664*u^9 - 197098733568*u^8 + 225723154432*u^7 - 200949432320*u^6 + 104049119232*u^5 + 18576939008*u^4 - 95299389856*u^3 + 84429624944*u^2 - 32502024478*u + 4592331303)*Du^4 + (90194313216*u^10 - 450971566080*u^9 + 1131254120448*u^8 - 1819187085312*u^7 + 2341711839232*u^6 - 2552061296640*u^5 + 1837924485120*u^4 - 642855276544*u^3 - 107272338384*u^2 + 171262804944*u - 40004119694)*Du^3 + (228170137600*u^9 - 1026765619200*u^8 + 2441311420416*u^7 - 3753017081856*u^6 + 5324394749952*u^5 - 6324230615040*u^4 + 4612935350016*u^3 - 1787170538112*u^2 + 295669884990*u - 5648844383)*Du^2 + (123480309760*u^8 - 493921239040*u^7 + 1075214024704*u^6 - 1496917737472*u^5 + 2495027740672*u^4 - 3071434031104*u^3 + 2030169438592*u^2 - 661618506112*u + 107476388194)*Du + 3355443200*u^7 - 11744051200*u^6 + 21705523200*u^5 - 24903680000*u^4 + 66255616000*u^3 - 80351769600*u^2 + 38163929600*u - 6240505600
chose

These matrices are defined only up to monodromy

In [47]:
swaps = [swapB1B2, swapB1B3, swapB1B4]
for swap in swaps:
    print(swap, end="\n\n")

[     1      0      0      0]
[     1      1      I -I + 1]
[     2      0      I -I + 1]
[-I + 1      0      0     -I]

[     1      1      0     -I]
[     0      1      0      0]
[     0  I + 1     -I      0]
[     0      0 -I - 1      I]

[     0      1      0      0]
[     1      2  I - 1 -I + 1]
[     0      2      I -I + 1]
[     0   -2*I  I + 1     -I]



In [48]:
swaps_on_TrX = [Lminus_to_eigenspaces.transpose() * block_diagonal_matrix([swap, swap.conjugate()]) * Lminus_to_eigenspaces.transpose().inverse() for swap in swaps]

In [49]:
swaps_on_discriminant = [matrix(IntegerModRing(2), [discriminant_group(swap*v.lift()) for v in discriminant_group.gens()]) for swap in swaps_on_TrX]
swaps_on_discriminant = [CBdiscriminant*M*CBdiscriminant.change_ring(IntegerModRing(2)).inverse() for M in swaps_on_discriminant]
for swap in swaps_on_discriminant:
    print(swap, end="\n\n")

[1 0 0 1 0 1]
[0 1 1 0 0 1]
[0 0 1 0 0 0]
[0 0 0 1 0 0]
[0 0 1 1 1 0]
[0 0 0 0 0 1]

[1 0 0 0 0 0]
[0 1 0 0 0 0]
[0 1 1 0 1 0]
[1 0 0 1 1 0]
[0 0 0 0 1 0]
[1 1 0 0 0 1]

[0 0 0 1 0 0]
[0 0 1 0 0 0]
[0 1 0 0 0 0]
[1 0 0 0 0 0]
[0 0 0 0 1 0]
[0 0 0 0 0 1]



The area permutations generate the group $S_4$

In [50]:
MatrixGroup(swaps_on_discriminant).group_id()

[24, 12]

The group action on the discriminant induced by monodromy and permutation of the $B_i$'s is $W(B_4)\simeq (\mathbb Z/2\mathbb Z)^4\rtimes S_4$.

In [51]:
MatrixGroup(monodromy_on_discriminant + swaps_on_discriminant).group_id()

[384, 5602]

Adding triality, we obtain $W(F_4)$

In [52]:
trialityMatrix = matrix(4, [1,0,0,0,0,1,0,0,0,0,0,-1,0,0,1,-1])
trialityMatrix = Lminus_to_eigenspaces.transpose() * block_diagonal_matrix([trialityMatrix, trialityMatrix.conjugate()]) * Lminus_to_eigenspaces.transpose().inverse()
trialityMatrix = matrix(IntegerModRing(2), [discriminant_group(trialityMatrix*v.lift()) for v in discriminant_group.gens()])
trialityMatrix = CBdiscriminant*trialityMatrix*CBdiscriminant.change_ring(IntegerModRing(2)).inverse()

In [53]:
MatrixGroup(monodromy_on_discriminant + swaps_on_discriminant + [trialityMatrix]).group_id()

[1152, 157478]

And the full orthogonal group of the discriminant is $W(E_6)$

In [54]:
discriminant_group.orthogonal_group().cardinality()

51840

In [55]:
WG = WeylGroup("E6")

In [56]:
WG.is_isomorphic(discriminant_group.orthogonal_group())

True

In [57]:
trialityMatrix

[1 0 0 0 0 0]
[0 1 0 0 0 0]
[0 0 1 0 0 0]
[0 0 0 1 0 0]
[0 0 0 0 0 1]
[0 0 0 0 1 1]